In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import time
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import string
import numpy as np
import tensorflow as tf
from keras.layers import (
    Dense,
    Convolution2D,
    MaxPooling2D,
    Flatten,
)
from keras.models import Sequential
from PIL import Image, ImageOps


In [ ]:
RUSSIAN_WEIGHTS = "api/weights/comnist_keras_ru.hdf5"
TEST_DIR = "images/Cyrillic_test"
SIZE = 32
nb_classes = 34
batch_size = 32
img_height = 32
img_width = 32



In [ ]:
# Load the datasets


# test_dataset = tf.keras.utils.image_dataset_from_directory(
#     TEST_DIR,
#     seed=123,
#     image_size=(img_height, img_width),
#     batch_size=batch_size,
#     labels="inferred",
# )

class_names = sorted([d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))])
name_to_index = {name: i for i, name in enumerate(class_names)}
print(class_names)
print(name_to_index)

In [ ]:
from api.model import load_letter_predictor
from IPython.display import display


commnist_predictor = load_letter_predictor(weight=RUSSIAN_WEIGHTS, nb_classes=34, lang_in="ru")
img_copy = Image.open("images/Cyrillic_test/А/5a2f3c19c27bb.png")
blank = Image.new("L", img_copy.size, color=255)
img_copy.paste(blank, (0, 0), mask=img_copy)
img_copy = img_copy.convert("L")
img_copy = ImageOps.invert(img_copy)
display(img_copy)


In [ ]:
from api.image_proc import crop_letters


for i, letter in enumerate(crop_letters(img_copy)):
    predicted_letter = commnist_predictor(letter,1)
    print(f"Predicted letter: {predicted_letter}")

In [ ]:
y_true = []
y_pred = []

for cls in class_names:
    folder = os.path.join(TEST_DIR, cls)
    for fn in os.listdir(folder):
        img_path = os.path.join(folder, fn)
        img = Image.open(img_path)

            # Add a white background to the image

        blank = Image.new("L", img.size, color=255)
        img.paste(blank, (0, 0), mask=img)
        img = img.convert("L")
        img = ImageOps.invert(img)

        # Get negative of image in case it is white on black
        # img_np = np.array(img)
        # if np.mean(img_np) < 128:
            
            
            
        pred_list = commnist_predictor(img, nb_output=1)   # returns list like ['Д']
        pred_char = pred_list[-1]
        # convert true/pred to integer class indices
        y_true.append(name_to_index[cls])
        # find predicted class index in class_names (ensure char == folder name)
        # if your folder names are letter characters use:
        y_pred.append(name_to_index.get(pred_char, -1))



In [ ]:

len(y_true), len(y_pred)
print("Accuracy:", accuracy_score(y_true, y_pred)*100,"%")
# print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)